# Week 09 — BBO capstone driver

Round 9. Two distinct experiments in one round.

**Mirror probes (F3, F4, F6).** Reflect the incumbent through the domain centre: x -> 1-x. If these surfaces carry any large-scale symmetry, the mirror should return something comparable. A cheap, falsifiable hypothesis with a clear negative result available in one round.

**Directional continuation (F2, F7, F8).** Step along the directions inferred from the W8 pairs. F8 gets a 10x nudge along its known-good direction.

**F1** goes somewhere new entirely — five rounds of readings within 10^-13 of zero mean the region is simply flat and nothing local will help.

In [ ]:
%matplotlib inline
import os, sys, warnings
warnings.filterwarnings("ignore")
# Walk up until bbo.py is found, so the notebook runs from anywhere in the repo.
_root = os.getcwd()
while not os.path.exists(os.path.join(_root, "bbo.py")) and os.path.dirname(_root) != _root:
    _root = os.path.dirname(_root)
os.chdir(_root); sys.path.insert(0, _root)
import numpy as np
import pandas as pd
import bbo

WEEK = 9
PRIOR = WEEK - 1          # data state this round was proposed from
SEED = 9
OUTDIR = f"outputs/week{WEEK:02d}"; os.makedirs(OUTDIR, exist_ok=True)

# What each function is getting this round, and why.
PLAN = {
    1: 'relocate — region flat',
    2: 'continue inferred direction',
    3: 'mirror probe (1-x)',
    4: 'mirror probe (1-x)',
    5: 'small local step',
    6: 'mirror probe (1-x)',
    7: 'continue inferred direction',
    8: '10x nudge along known direction',
}
pd.DataFrame([dict(func=f"F{f}", d=bbo.DIMS[f], move=PLAN[f]) for f in bbo.FUNC_IDS])


## 1. Data — the state this round was proposed from

Best point on record per function, truncated to rounds ≤ 8. Nothing below this cell may look at later rounds.


In [ ]:
# The ledger comes FIRST every round: the best point on record, not the latest one.
led = bbo.ledger(up_to=PRIOR)
led["best"] = led["best"].map(lambda v: f"{v:.6g}")
led["x"] = led["x"].map(bbo.submission)
led


## 2. Proposals — mirrors and continuations

The mirror check below confirms which vectors are exact reflections of their W8 incumbents.

In [ ]:
proposals = {
    1: np.array([0.330063, 0.371452]),
    2: np.array([0.654432, 0.749899]),
    3: np.array([0.888725, 0.225684, 0.483049]),
    4: np.array([0.816128, 0.934647, 0.982382, 0.095674]),
    5: np.array([0.322806, 0.716874, 0.106998, 0.647917]),
    6: np.array([0.929979, 0.469268, 0.047147, 0.174195, 0.771203]),
    7: np.array([0.686227, 0.361566, 0.535476, 0.622864, 0.339771, 0.580055]),
    8: np.array([0.039233, 0.275235, 0.182077, 0.347603, 0.803294, 0.234613, 0.936241, 0.094463]),
}

prev = {fid: np.array(bbo.HISTORY[8][fid][0]) for fid in bbo.FUNC_IDS}
pd.DataFrame([dict(func=f"F{fid}",
                   is_mirror=bool(np.allclose(proposals[fid], 1-prev[fid], atol=1e-6)),
                   step=round(float(np.linalg.norm(proposals[fid]-prev[fid])), 4),
                   submission=bbo.submission(proposals[fid]))
              for fid in bbo.FUNC_IDS])


### Surrogate trust check

Run before reading any acquisition value, not after.


In [ ]:
# Is each surrogate worth listening to? LOO R2 < 0 means it is worse than
# predicting the mean, and any acquisition value built on it is arbitrary.
rows = []
for fid in bbo.FUNC_IDS:
    X, y, _ = bbo.load(fid, up_to=PRIOR)
    r2 = bbo.fit(fid, up_to=PRIOR).loo_r2() if len(y) >= 4 else float("nan")
    rows.append(dict(func=f"F{fid}", n_data=len(y), loo_r2=round(r2, 3),
                     verdict="broken" if r2 < 0 else "usable" if r2 == r2 else "too few points"))
pd.DataFrame(rows)


### Anchor audit


In [ ]:
ANCHOR = {
    1: [0.669937, 0.751452],
    2: [0.354432, 0.449899],
    3: [0.111275, 0.774316, 0.516951],
    4: [0.183872, 0.065353, 0.017618, 0.904326],
    5: [0.308806, 0.730874, 0.091998, 0.661917],
    6: [0.070021, 0.530732, 0.952853, 0.825805, 0.228797],
    7: [0.965568, 0.153915, 0.588691, 0.807159, 0.099427, 0.700138],
    8: [0.038733, 0.275735, 0.181777, 0.348003, 0.803094, 0.234913, 0.935841, 0.094663],
}
# Anchor audit: is each proposal being generated from the best point on record?
# This is the check whose absence cost the campaign most of its final score.
for fid in bbo.FUNC_IDS:
    w = bbo.anchor_check(fid, np.array(ANCHOR[fid], float), up_to=PRIOR)
    print(f"F{fid}: {w if w else 'anchored on best-known point'}")


## 3. Visualise

Best-so-far trajectory per function, truncated to the data available this round.


In [ ]:
import matplotlib
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 4, figsize=(15, 6))
for ax, fid in zip(axes.ravel(), bbo.FUNC_IDS):
    try:
        _, y, rounds = bbo.load(fid, up_to=PRIOR)
    except ValueError:
        ax.set_title(f"F{fid}: no data"); continue
    ax.plot(rounds, y, "o", ms=4, alpha=.55)
    ax.plot(rounds, np.maximum.accumulate(y), "-", lw=2)
    ax.set_title(f"F{fid} (d={bbo.DIMS[fid]})", fontsize=9)
    ax.tick_params(labelsize=7); ax.set_xlabel("round", fontsize=8)
fig.suptitle(f"Best so far through round {PRIOR}", fontsize=11)
fig.tight_layout(); fig.savefig(f"{OUTDIR}/trajectories.png", dpi=140)
plt.show()


## 4. Submission strings


In [ ]:
# Portal format: six decimals, dash-separated, one line per function, no labels.
for fid in bbo.FUNC_IDS:
    print(bbo.submission(proposals[fid]))


## 5. After the portal returns each y

Returns recorded below and folded into `bbo.HISTORY` so the next round sees them.


In [ ]:
# Week 9 portal returns - already folded into bbo.HISTORY.
# returned_y = {
#     1: 1.04e-08,
#     2: 0.423255,
#     3: -0.035188,
#     4: -36.0907,
#     5: 0.309847,
#     6: -2.4701,
#     7: 0.416839,
#     8: 8.749654,
# }
#
# Symmetry hypothesis: FALSIFIED, cleanly. Every mirror came back worse -
#   F3 -0.0352 vs -0.0154, F4 -36.09 vs -23.62, F6 -2.470 vs -0.968.
# Worst round on the scoreboard, and money well spent: it killed a hypothesis that
# would otherwise have eaten later rounds.
# Continuations paid: F2 0.0837 -> 0.4233, F7 0.0798 -> 0.4168. Directions work.
# F1 1.04e-8 - best F1 of the project, and still essentially zero.
#
# for fid, y in returned_y.items():
#     bbo.append_result(fid, proposals[fid], y, rnd=WEEK)
